# 3章 サンプル（PyTorch版） ― 16×16ドットで「1」を見分ける

TrainZu2Pt.pyのColab向け版。オリジナル版はdata2フォルダの10ファイルをそのまま読み込む形
だったため、Colabでもそのまま動くようにデータをリテラルに変えている（03-2Train2.ipynbと
同じ10件のデータ）。手作り版（03-2）と同じ問題を、PyTorchの`nn.Linear`と`optimizer`に
置き換えて解いている。

なお、オリジナル版はラベルと予測対象の対応が「1なら0、1でなければ1」という分かりにくい
向きになっていたため、ここでは03-2・trainZu2G.pyと同じ「1なら1、1でなければ0」という
向きに揃えている。


In [ ]:
import torch
import torch.nn as nn

# 03-2Train2.ipynbと同じ10件のデータ
RAW_DATA = [
    ("0_01.txt", [
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "...■■■■■■■■■■...",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
    ]),
    ("0_02.txt", [
        "................",
        "................",
        "....■■■■■■■■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■■■■■■■■....",
        "................",
        "................",
        "................",
        "................",
    ]),
    ("0_03.txt", [
        "................",
        "................",
        "...■............",
        "....■...........",
        ".....■..........",
        "......■.........",
        ".......■........",
        "........■.......",
        ".........■......",
        "..........■.....",
        "...........■....",
        "............■...",
        "................",
        "................",
        "................",
        "................",
    ]),
    ("0_04.txt", [
        "................",
        "................",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        "...■■■■■■■■■....",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        "................",
        "................",
        "................",
    ]),
    ("0_05.txt", [
        "................",
        "................",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "................",
        "................",
        "................",
        "................",
    ]),
    ("1_01.txt", [
        ".......■........",
        "......■■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        "......■■■.......",
        "................",
    ]),
    ("1_02.txt", [
        "......■.........",
        ".....■■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        ".....■■■........",
        "................",
    ]),
    ("1_03.txt", [
        "........■.......",
        ".......■■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        ".......■■■......",
        "................",
    ]),
    ("1_04.txt", [
        ".......■........",
        ".....■■■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        "......■■■.......",
        "................",
    ]),
    ("1_05.txt", [
        ".......■........",
        "......■■........",
        ".....■.■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".....■■■■■......",
        "................",
    ]),
]


## データを読み込む

In [ ]:
class TrainingData:
    def __init__(self, label, bits, filename):
        self.label, self.bits, self.filename = label, bits, filename


def load_sample(filename, rows):
    if len(rows) != 16 or any(len(row) != 16 for row in rows):
        raise ValueError(filename + " は16文字×16行にしてください")
    bits = [[ch == "■" for ch in row] for row in rows]
    label = filename.startswith("1_")
    return TrainingData(label, bits, filename)


TrainingDatas = [load_sample(name, rows) for name, rows in RAW_DATA]


# bitsを256個のTensorに変換
def bits_to_tensor(bits):
    values = []
    for y in range(16):
        for x in range(16):
            values.append(1.0 if bits[y][x] else 0.0)
    return torch.tensor(values, dtype=torch.float32)


# 判定対象のビット配列から予測値を求める
def predict(bits):
    input_tensor = bits_to_tensor(bits)
    return linear(input_tensor)


## モデル（PyTorchの`nn.Linear`で256→1）

In [ ]:
# 16×16 = 256個の入力値から、1個の予測値を出す
linear = nn.Linear(256, 1)

# 手作り版と同じように weight と bias を 0 から開始
with torch.no_grad():
    linear.weight.zero_()
    linear.bias.zero_()

print("===== 学習前 =====")
for s in TrainingDatas:
    prediction = predict(s.bits)
    print(s.filename, "教師=", s.label, "非合致度=", round(prediction.item(), 4))


## 学習ループ

In [ ]:
learning_rate = 0.01
epochs = 1000

loss_function = nn.MSELoss()
optimizer = torch.optim.SGD(linear.parameters(), lr=learning_rate)

for epoch in range(epochs):
    total_loss = 0.0

    for s in TrainingDatas:
        # ①予測
        prediction = predict(s.bits)

        # 1なら target=1、1でなければ target=0（03-2・trainZu2G.pyと同じ向き）
        target = torch.tensor([1.0 if s.label else 0.0], dtype=torch.float32)

        optimizer.zero_grad()

        # ②Loss
        loss = loss_function(prediction, target)
        total_loss += loss.item()

        # ③Lossを微分
        loss.backward()

        # ④⑤weightとbiasを補正
        optimizer.step()

    if epoch == 0 or (epoch + 1) % 100 == 0:
        print("epoch =", epoch + 1, "loss =", round(total_loss / len(TrainingDatas), 6))


In [ ]:
print("\n===== 学習後 =====")
for s in TrainingDatas:
    prediction = predict(s.bits)
    print(s.filename, "教師値=", s.label, "非合致度=", round(prediction.item(), 4))


## 未知の画像で試す

03-2Train2.ipynbと同じ、学習データにない「1」の形を判定させる。

In [ ]:
test = [[False]*16 for _ in range(16)]
test[1][7] = True
test[2][6] = test[2][7] = True
for y in range(3,14):
    test[y][7] = True
test[14][6] = test[14][7] = test[14][8] = True

print("===== 未知画像 =====")
for row in test:
    print("".join("■" if v else " " for v in row))

p = predict(test).item()
print("\n1である予測値 =", round(p, 4))
print("判定 =", "1" if p >= 0.5 else "1ではない")
